# `subprocess` — calling other programs from Python

## The problem it solves

Sometimes the tool you need isn't a Python function — it's an entirely
separate program: `ffmpeg`, `pg_dump`, `git`, a shell script, a legacy
Perl pipeline that nobody is rewriting. Python can't call that code
directly, because it isn't Python.

`subprocess` lets your program launch another program as a **child
process**, optionally feed it input, and get back three things once
it's done: what it printed (**stdout**), what it complained about (**stderr**),
and whether it succeeded (the **exit/return** code).

Before `subprocess` existed, people used `os.system()` (no way to
capture output) or `os.popen()` (clunky, inconsistent). `subprocess`
unified all of that into one module — it's the modern, only-correct way
to do this in Python 3.

## Analogy

Think of `subprocess` like phoning a contractor to do a job instead of
doing it yourself: you dial the number with specific instructions (the
command + its arguments), you either wait on the line for them to
finish or hang up and check back later, and at the end you get a
verdict — "job done" or "failed, here's why."

You already do something conceptually similar in your own code: a
Django view calling `requests.post()` to talk to another *service*.
`subprocess` is the same idea, except the "other service" is a program
sitting right there on the same machine.


In [1]:
import subprocess
import sys

# sys.executable instead of the literal string "python" — guarantees
# we call the *same* interpreter (and venv) this notebook is running on
result = subprocess.run(
    [sys.executable, "-c", "print('hello from subprocess')"],
    capture_output=True,
    text=True,
)

print("return code:", result.returncode)
print("stdout:", repr(result.stdout))
print("stderr:", repr(result.stderr))


return code: 0
stdout: 'hello from subprocess\n'
stderr: ''


In [2]:
sys.executable

'c:\\Users\\Dennie\\Mate Academy\\Python Core\\python_essentials\\.venv\\Scripts\\python.exe'

### What just happened

`subprocess.run(...)` is the high-level, "just do it and wait" entry
point — the one you reach for 95% of the time.

- The first argument is a **list**: `[program, arg1, arg2, ...]`. Each
  element is one argument, the same way `sys.argv` would split it —
  there is no shell involved parsing a string.
- `capture_output=True` redirects the child's stdout/stderr into pipes
  instead of letting them print straight to your terminal.
- `text=True` decodes those pipes as `str` instead of raw `bytes`.
- `result.returncode` follows the universal OS convention: `0` means
  success, anything else means failure (the *meaning* of non-zero
  codes is program-specific).

> **It blocks** — your notebook cell waits until the child process exits
before the next line runs.


In [3]:
import sys

sys.exit(1)

SystemExit: 1

c:\Users\Dennie\Mate Academy\Python Core\python_essentials\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [4]:
import subprocess
import sys

# check=True tells subprocess to raise instead of silently returning
# a non-zero returncode — the same instinct as checking response.ok
# on a requests call instead of ignoring the status code
try:
    subprocess.run(
        [sys.executable, "-c", "import sys; sys.exit(1)"],
        capture_output=True,
        text=True,
        check=True,
    )
except subprocess.CalledProcessError as exc:
    print(f"command failed with exit code {exc.returncode}")
    print(f"stderr: {exc.stderr!r}")


command failed with exit code 1
stderr: ''


## Where this actually comes up in backend work

```shell
Scenario                                          → Why subprocess
─────────────────────────────────────────────────────────────────────
Django management command backing up the DB      → calls `pg_dump`
Celery task transcoding an uploaded video         → calls `ffmpeg`
Generating a PDF report from HTML                 → calls `wkhtmltopdf` / pandoc
Custom CI / pre-commit tooling                     → calls `ruff`, `mypy`, `black`
A legacy data pipeline nobody is rewriting          → calls a bash/Perl/R script
Deploy script checking out a tag before building   → calls `git`
```

The common thread: the *logic* doesn't exist in Python, but the
*orchestration* (when to call it, what to do with the result) does.


In [ ]:
import subprocess
import sys
from pathlib import Path


def export_database_backup(output_path: Path) -> None:
    """
    Shape of a real Django management command that shells out to
    `pg_dump`. In production the command list would be:

        ["pg_dump", "-Fc", "-f", str(output_path), "mydb"]

    Here we stand in for `pg_dump` with the current interpreter so the
    cell runs anywhere, but the *pattern* — build args as a list, run
    it, check returncode, raise a domain error — is identical.
    """
    fake_pg_dump = [
        sys.executable,
        "-c",
        f"open(r'{output_path}', 'w').write('-- fake backup --')",
    ]
    result = subprocess.run(fake_pg_dump, capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(f"backup failed: {result.stderr}")

    print(f"backup written to {output_path}")


export_database_backup(Path("backup.sql"))


## `run()` vs `Popen` — what's actually underneath

`subprocess.run()` is not a separate mechanism — it's a thin,
blocking wrapper *around* `subprocess.Popen`. `Popen` is the real
primitive: it starts the child and immediately hands control back to
you, while the child keeps running in the background.

That matters when:
- the child is long-running and you want to **stream** its output as
  it produces it (progress logs from a training job, a build, an
  ffmpeg conversion), instead of waiting for it to finish entirely
- you need to send input to it interactively, or kill it early
- you want to poll `.poll()` without blocking

Under the hood: on POSIX, Python `fork()`s the process and then
`exec()`s the target program in the child; on Windows it's
`CreateProcess`. If you asked for `stdout=PIPE`, the OS gives you a
pipe (a small kernel buffer) connecting the child's stdout to a file
descriptor you can read from in the parent.


In [ ]:
import subprocess
import sys

# a child that prints progress over time, line by line —
# stand-in for a Celery task shelling out to a long-running tool
script = (
    "import time\n"
    "for i in range(3):\n"
    "    print(f'step {i}', flush=True)\n"
    "    time.sleep(0.5)\n"
)

process = subprocess.Popen(
    [sys.executable, "-c", script],
    stdout=subprocess.PIPE,
    text=True,
)

# iterating the pipe yields lines as the child produces them —
# we don't wait for the whole process to finish first
for line in process.stdout:
    print(f"[child] {line.rstrip()}")

process.wait()
print("exit code:", process.returncode)


## Gotchas — what breaks this in real code

- **`shell=True` + string-built commands = command injection.** This
  is the `subprocess` equivalent of building raw SQL with f-strings.
  If any part of the command string comes from a user, they can
  smuggle in extra commands. Passing a **list** of arguments
  (`shell=False`, the default) skips shell parsing entirely — there's
  nothing to inject into.
- **PIPE deadlock.** If you manually wire `stdout=PIPE` *and*
  `stderr=PIPE` and only drain one of them yourself while the child
  fills up the other, the OS pipe buffer fills and the child hangs
  forever waiting for someone to read it. `run()` and
  `Popen.communicate()` avoid this by draining both streams
  concurrently — don't reinvent that loop unless you need to stream.
- **Zombie processes.** With raw `Popen`, if you never call
  `.wait()` / `.communicate()` / `.poll()` until exit, the OS can be
  left holding onto the finished child's exit status (a "zombie") on
  POSIX systems.
- **It blocks.** `subprocess.run()` blocks the calling thread until
  the child exits. Calling it inside an `async def` FastAPI endpoint
  freezes the entire event loop — every other concurrent request
  waits too. Use `asyncio.create_subprocess_exec` (or
  `run_in_executor`) in async code instead.


In [ ]:
import subprocess
import sys

# DANGEROUS — never build a shell command from untrusted input.
# Imagine `filename` came from a request parameter:
filename = "report.txt & echo INJECTED"
subprocess.run(f"type {filename}", shell=True)  # `type` is Windows' `cat`
# The shell sees TWO commands separated by `&` and happily runs both —
# whatever the attacker put after `&` executes with your process's privileges.

print("-" * 40)

# SAFE — arguments as a list, no shell string parsing involved.
# Even with shell=False (the default), the same "filename" is just one
# literal argument — there is nothing for it to "break out" of.
subprocess.run([sys.executable, "-c", "print('this stays safe')"], shell=False)


## Interview angle

**Question:** "What is `subprocess` used for, and what's the danger
of `shell=True`?"

```
Level 1 (passes):
  "It spawns external programs and lets you capture their
   stdout/stderr and exit code."

Level 2 (good candidate):
  "subprocess.run() is the blocking, high-level call — check=True
   raises CalledProcessError on a non-zero exit instead of letting
   you silently ignore failures. Popen is the lower-level primitive
   for streaming or long-running processes."

Level 3 (strong candidate):
  "shell=True passes the command through a real shell, so any
   user-controlled string in it is a command-injection vector —
   the same class of bug as building raw SQL with string
   concatenation. Always shell=False with a list of args for
   untrusted input. Also: subprocess.run blocks, so it can't be
   called directly inside async code without freezing the event
   loop — asyncio.create_subprocess_exec is the async-native version."
```

**Follow-up the interviewer asks next:** "How would you run this
without blocking a FastAPI event loop?" → `asyncio.create_subprocess_exec`
or offload via `loop.run_in_executor`.

**The trap:** saying "`shell=True` is just for convenience" without
flagging that it's a security boundary, not a style preference.


## Try it yourself

Write a function `convert_video(input_path, output_path)` that shells
out to `ffmpeg` (or `sys.executable` as a stand-in, like above) to do
the conversion. It should:

1. Raise a custom `ConversionError` with the tool's stderr message if
   the process exits non-zero.
2. **Bonus:** use `Popen` instead of `run()` to stream progress lines
   as they're produced, instead of blocking until the whole
   conversion finishes.
